# 手写数字识别

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [11]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [12]:
train_dataset = datasets.MNIST(
    root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(
    root='./data', train=False, download=True, transform=transform)

In [13]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [14]:
class mnistModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")  # Set the device to CUDA, MPS, or CPU
print(f"Using device: {device}")
model = mnistModel().to(device)  # Move the model to the selected device (GPU, MPS, or CPU)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Using device: mps


In [20]:
def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()
    running_loss = 0.0
    #correct是当前批次的正确预测的样本数，total是总样本数
    correct = 0
    total = 0
#这里的enumerate(dataloader)会返回两个值：batch_idx是当前批次的索引，(data, target)是当前批次的数据和标签。
    for batch_idx, (data, target) in enumerate(dataloader):
        data, target = data.to(device), target.to(device)  # Move the data and target to the selected device

        optimizer.zero_grad()  # Clear the gradients
        output = model(data)  # Forward pass
        loss = loss_fn(output, target)  # Compute the loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update the model parameters

        # track the running loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(output.data, 1)  # Get the index of the max log-probability
        total += target.size(0)  # Update the total number of samples
        correct += (predicted == target).sum().item()  # Update the

        # print the loss and accuracy every 100 batches
        if batch_idx % 100 == 99:
            print(f'Batch {batch_idx + 1}, Loss: {running_loss / 100:.4f}, Accuracy: {100 * correct / total:.2f}%')
            running_loss = 0.0
            correct = 0
            total = 0

In [21]:
def evaluate(model, test_dataset, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_dataset:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    return 100 * correct / total

In [23]:
num_epochs = 5

for epoch in range(num_epochs):
    print(f'Epoch {epoch + 1}/{num_epochs}')
    train_epoch(model, train_loader, loss_fn, optimizer, device)
    accuracy = evaluate(model, test_loader, device)
    print(f'Test Accuracy: {accuracy:.2f}%\n')

Epoch 1/5
Batch 100, Loss: 0.0044, Accuracy: 99.89%
Batch 200, Loss: 0.0048, Accuracy: 99.83%
Batch 300, Loss: 0.0080, Accuracy: 99.66%
Batch 400, Loss: 0.0121, Accuracy: 99.62%
Batch 500, Loss: 0.0071, Accuracy: 99.73%
Batch 600, Loss: 0.0092, Accuracy: 99.73%
Batch 700, Loss: 0.0147, Accuracy: 99.61%
Batch 800, Loss: 0.0118, Accuracy: 99.67%
Batch 900, Loss: 0.0099, Accuracy: 99.67%
Test Accuracy: 97.97%

Epoch 2/5
Batch 100, Loss: 0.0091, Accuracy: 99.70%
Batch 200, Loss: 0.0053, Accuracy: 99.80%
Batch 300, Loss: 0.0034, Accuracy: 99.86%
Batch 400, Loss: 0.0078, Accuracy: 99.81%
Batch 500, Loss: 0.0049, Accuracy: 99.83%
Batch 600, Loss: 0.0035, Accuracy: 99.91%
Batch 700, Loss: 0.0030, Accuracy: 99.94%
Batch 800, Loss: 0.0032, Accuracy: 99.88%
Batch 900, Loss: 0.0026, Accuracy: 99.88%
Test Accuracy: 98.00%

Epoch 3/5
Batch 100, Loss: 0.0037, Accuracy: 99.88%
Batch 200, Loss: 0.0065, Accuracy: 99.84%
Batch 300, Loss: 0.0053, Accuracy: 99.83%
Batch 400, Loss: 0.0064, Accuracy: 99.80%
